# Решения: Проект 19 — рынок труда IT в России

Решения упражнений из `19_IT_Job_Market_Russia.ipynb`. Данные (снимок
hh.ru/Habr Career/ЦИАН/Росстат) и базовая модель воспроизведены ниже.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

SNAPSHOT = {
  'Москва':(250000,60000,340000,130000), 'Санкт-Петербург':(210000,38000,230000,95000),
  'Новосибирск':(180000,28000,130000,68000), 'Екатеринбург':(175000,30000,130000,70000),
  'Казань':(175000,30000,175000,65000), 'Иннополис':(180000,32000,165000,65000),
  'Нижний Новгород':(165000,28000,140000,60000), 'Краснодар':(160000,30000,150000,58000),
  'Ростов-на-Дону':(155000,27000,120000,55000), 'Самара':(150000,25000,110000,56000),
  'Воронеж':(150000,24000,100000,52000), 'Пермь':(150000,24000,110000,58000),
  'Уфа':(150000,25000,120000,56000), 'Красноярск':(165000,27000,120000,72000),
  'Челябинск':(150000,24000,105000,58000), 'Тюмень':(165000,30000,130000,78000),
  'Томск':(150000,25000,110000,60000), 'Калининград':(155000,28000,140000,55000),
  'Владивосток':(170000,35000,200000,75000), 'Волгоград':(140000,22000,95000,50000),
  'Омск':(140000,22000,100000,55000), 'Сочи':(160000,50000,280000,60000),
}
df = pd.DataFrame([(c0, v[0], v[1], v[2], v[3]) for c0, v in SNAPSHOT.items()],
                  columns=['city','it_salary','rent','price_m2','avg_salary'])

NDFL_BRACKETS = [(0,2_400_000,0.13),(2_400_000,5_000_000,0.15),(5_000_000,20_000_000,0.18),
                 (20_000_000,50_000_000,0.20),(50_000_000,np.inf,0.22)]
def ndfl(g):
    tax = 0.0
    for lo, hi, rate in NDFL_BRACKETS:
        if g > lo: tax += (min(g, hi) - lo) * rate
        else: break
    return tax

IT_MORTGAGE_RATE, DOWN, YEARS, AREA = 0.06, 0.20, 30, 50
def annual_mortgage(price_m2, rate=IT_MORTGAGE_RATE):
    P = price_m2 * AREA * (1 - DOWN); r = rate / 12; n = YEARS * 12
    return P * r * (1 + r) ** n / ((1 + r) ** n - 1) * 12

def build(df, sal_col='it_salary'):
    df = df.copy()
    df['gross_year'] = df[sal_col] * 12
    df['net_year'] = df['gross_year'] - df['gross_year'].apply(ndfl)
    df['net_month'] = df['net_year'] / 12
    df['home_price'] = df['price_m2'] * AREA
    df['annual_rent'] = df['rent'] * 12
    df['annual_mortgage'] = df['price_m2'].apply(annual_mortgage)
    df['dispo_rent'] = df['net_year'] - df['annual_rent']
    df['dispo_buy'] = df['net_year'] - df['annual_mortgage']
    df['price_to_income'] = df['home_price'] / df['gross_year']
    return df

def zscore(s): return (s - s.mean()) / s.std(ddof=0)
WEIGHTS = {'net_year':0.40,'dispo_rent':0.30,'annual_rent':-0.15,'price_to_income':-0.15}
def compute_score(d, w):
    s = pd.Series(0.0, index=d.index)
    for col, wt in w.items(): s += wt * zscore(d[col])
    return s

df = build(df)
df['score'] = compute_score(df, WEIGHTS)
print('Топ-5 (базовый рейтинг):')
print(df.sort_values('score', ascending=False)[['city','net_month','dispo_rent']].head())

## Упражнение 1: Грейды junior / middle / senior

Зарплаты снимка соответствуют уровню middle. Добавим множители и посмотрим рейтинг senior.

In [ ]:
GRADES = {'junior': 0.55, 'middle': 1.0, 'senior': 1.7}
for grade, mult in GRADES.items():
    d = df[['city','rent','price_m2','avg_salary']].copy()
    d['it_salary'] = df['it_salary'] * mult
    d = build(d); d['score'] = compute_score(d, WEIGHTS)
    top3 = d.sort_values('score', ascending=False)['city'].head(3).tolist()
    print(f'{grade:<7} (×{mult}): топ-3 -> {top3}')

# Детально по senior
ds = df[['city','rent','price_m2','avg_salary']].copy()
ds['it_salary'] = df['it_salary'] * GRADES['senior']
ds = build(ds); ds['score'] = compute_score(ds, WEIGHTS)
top = ds.sort_values('score', ascending=False)[['city','net_month','dispo_rent']].head(6).copy()
top['net_month'] = top['net_month'].map('{:,.0f} ₽'.format)
top['dispo_rent'] = top['dispo_rent'].map('{:,.0f} ₽'.format)
print('\nТоп-6 для SENIOR:')
print(top.to_string(index=False))
print('\nУ senior выше номинал -> прогрессия НДФЛ (15%+) чуть заметнее,')
print('но города-лидеры устойчивы (Москва/Новосибирск/Иннополис).')

## Упражнение 2: Полные расходы (коммуналка + транспорт)

Добавим оценку ЖКХ и транспорта в месяц по городам (крупные города дороже) и
пересчитаем располагаемый доход.

In [ ]:
# Приблизительные ежемесячные расходы: ЖКХ + транспорт (₽/мес)
BIG = {'Москва':13000,'Санкт-Петербург':11000,'Владивосток':10000,'Сочи':10000}
d = df.copy()
d['utilities'] = d['city'].map(BIG).fillna(8000)   # по умолчанию ~8000 ₽/мес
d['annual_extra'] = d['utilities'] * 12
d['dispo_full'] = d['dispo_rent'] - d['annual_extra']

top = d.sort_values('dispo_full', ascending=False)[
    ['city','dispo_rent','utilities','dispo_full']].head(8).copy()
top['dispo_rent'] = top['dispo_rent'].map('{:,.0f}'.format)
top['utilities'] = top['utilities'].map('{:,.0f}'.format)
top['dispo_full'] = top['dispo_full'].map('{:,.0f}'.format)
print('Топ-8 по располагаемому доходу с учётом ЖКХ+транспорта (₽/год):')
print(top.to_string(index=False))
print('\nПоправка на коммуналку/транспорт снижает абсолютные суммы,')
print('но региональные лидеры почти не меняются.')

## Упражнение 3: Аренда vs IT-ипотека

Рейтинг по `dispo_buy` и ставка безубыточности (годовая ипотека = годовой аренде).

In [ ]:
top_buy = df.sort_values('dispo_buy', ascending=False)[
    ['city','dispo_rent','annual_mortgage','dispo_buy']].head(8).copy()
for col in ['dispo_rent','annual_mortgage','dispo_buy']:
    top_buy[col] = top_buy[col].map('{:,.0f}'.format)
print('Топ-8 по располагаемому доходу при ПОКУПКЕ (IT-ипотека 6%):')
print(top_buy.to_string(index=False))

# Ставка безубыточности для Москвы и Новосибирска
for city in ['Москва', 'Новосибирск']:
    row = df[df.city == city].iloc[0]
    def gap(rate): return annual_mortgage(row['price_m2'], rate) - row['annual_rent']
    lo, hi = 0.0001, 0.30
    for _ in range(60):
        mid = (lo + hi) / 2
        if gap(mid) > 0: hi = mid
        else: lo = mid
    print(f'{city}: покупка выгоднее аренды при ставке < {mid*100:.2f}% '
          f'(ипотека 6%: {annual_mortgage(row["price_m2"]):,.0f} vs аренда {row["annual_rent"]:,.0f} ₽/год)')

## Упражнение 4: Форма занятости при разных доходах

Сравним доход на руки для наёмного (НДФЛ), самозанятого (НПД 6%) и ИП УСН 6%
на разных уровнях годового дохода и найдём точки перехода.

In [ ]:
FIXED_CONTRIB = 53000
def net_employee(g): return g - ndfl(g)
def net_selfemp(g): return g * 0.94                      # НПД 6% (лимит 2.4 млн)
def net_ip_usn(g):
    usn = g * 0.06
    extra = max(0, g - 300000) * 0.01
    return g - max(usn, FIXED_CONTRIB + extra)

incomes = np.arange(600_000, 8_000_001, 100_000)
emp = [net_employee(g) for g in incomes]
se = [net_selfemp(g) for g in incomes]
ip = [net_ip_usn(g) for g in incomes]

plt.figure(figsize=(11, 6))
plt.plot(incomes/1e6, np.array(emp)/1e6, label='Наёмный (НДФЛ)', lw=2)
plt.plot(incomes/1e6, np.array(se)/1e6, label='Самозанятый (НПД 6%)', lw=2)
plt.plot(incomes/1e6, np.array(ip)/1e6, label='ИП УСН 6%', lw=2)
plt.axvline(2.4, color='gray', ls='--', alpha=0.6, label='лимит самозанятости 2.4 млн')
plt.xlabel('Годовой доход (млн ₽)'); plt.ylabel('На руки (млн ₽)')
plt.title('Доход на руки vs форма занятости'); plt.legend()
plt.tight_layout(); plt.show()

# Где НПД перестаёт быть лучшим (лимит 2.4 млн)
print('До 2.4 млн ₽/год самозанятость (НПД 6%) и ИП УСН выгоднее наёмного (НДФЛ 13%+).')
print('Свыше лимита 2.4 млн самозанятость недоступна -> остаётся ИП УСН или наёмный.')
g_hi = 6_000_000
print(f'\nПример дохода {g_hi:,} ₽/год:')
print(f'  Наёмный:      {net_employee(g_hi):,.0f} ₽ (эфф. {(1-net_employee(g_hi)/g_hi)*100:.1f}%)')
print(f'  ИП УСН 6%:    {net_ip_usn(g_hi):,.0f} ₽ (эфф. {(1-net_ip_usn(g_hi)/g_hi)*100:.1f}%)')

## Упражнение 5: Свои приоритеты (доступность покупки жилья важнее всего)

In [ ]:
MY_WEIGHTS = {'net_year':0.20,'dispo_rent':0.20,'annual_rent':-0.15,'price_to_income':-0.45}
d = df.copy(); d['my_score'] = compute_score(d, MY_WEIGHTS)
top5 = d.sort_values('my_score', ascending=False)[
    ['city','net_month','rent','price_to_income']].head(5).copy()
top5['net_month'] = top5['net_month'].map('{:,.0f} ₽'.format)
top5['rent'] = top5['rent'].map('{:,.0f} ₽'.format)
top5['price_to_income'] = top5['price_to_income'].map('{:.1f} лет'.format)
print('Мой топ-5 (приоритет — доступность покупки жилья):')
print(top5.to_string(index=False))
print('\nВперёд выходят города с дешёвым жильём относительно зарплаты')
print('(Новосибирск, Челябинск, Волгоград, Омск).')